<center><h2><b>Natural Language Processing (NLP)</b></h2>
<h3><b>Notebook 1: Arabic Text Preprocessing — AAFAQ Dataset</b></h3>
<h4>Student: Hala Nazzal</h4><hr></center>

<h4>Before we start, install the required packages:</h4>

```
!pip install nltk pyarabic pandas
```


In [5]:
!pip install nltk pyarabic -q


In [6]:
import re
import nltk
import pandas as pd
import pyarabic.araby as araby
from nltk.corpus import stopwords
from nltk.stem.isri import ISRIStemmer
from nltk.tokenize import word_tokenize

nltk.download('stopwords', quiet=True)
nltk.download('punkt',     quiet=True)
nltk.download('punkt_tab', quiet=True)


True

### Load Dataset
---

In [7]:
df = pd.read_csv('/content/AAFAQ_Dataset.csv', encoding='utf-8-sig')
print(f"Shape: {df.shape}")
df.head()


Shape: (5009, 3)


,QuestionText,Category,Answer
0,أيهما أفضل الدراسة في السابق أم في الوقت الحالي؟,التعليم,الدراسة في الوقت الحالي تعتبر أفضل بسبب توفر ا...
1,أليس القطن عماد الثروة في مصر؟,الاقتصاد والعمل,القطن يعتبر من أهم المنتجات الزراعية في مصر، و...
2,أتصعد الشمس من الشرق؟,التعليم,الشمس تصعد من الشرق.
3,أتعرف البكتيريا بأنها كائنات حية دقيقة؟,التعليم,البكتيريا تُعرف بأنها كائنات حية دقيقة.
4,أيتكون الهواء أساساً من النيتروجين؟,التعليم,الهواء يتكون أساساً من النيتروجين.


In [8]:
print("Columns:", df.columns.tolist())
print("\nCategory distribution:")
print(df['Category'].value_counts())
df['Question']=df['QuestionText']
df.head()
df.drop(columns=['QuestionText'], inplace=True)
df.head()

Columns: ['QuestionText', 'Category', 'Answer']

Category distribution:
Category
الثقافة             765
العلوم              632
التعليم             624
الصحة               552
البيئة والطاقة      508
التكنولوجيا         454
التاريخ             418
الاقتصاد والعمل     298
الجغرافيا           212
السياسة والقانون    201
الدين               114
الرياضة              67
البيولوجيا           52
الترفيه              40
السفر والسياحة       37
التطوع               26
علم الاجتماع          9
Name: count, dtype: int64


,Category,Answer,Question
0,التعليم,الدراسة في الوقت الحالي تعتبر أفضل بسبب توفر ا...,أيهما أفضل الدراسة في السابق أم في الوقت الحالي؟
1,الاقتصاد والعمل,القطن يعتبر من أهم المنتجات الزراعية في مصر، و...,أليس القطن عماد الثروة في مصر؟
2,التعليم,الشمس تصعد من الشرق.,أتصعد الشمس من الشرق؟
3,التعليم,البكتيريا تُعرف بأنها كائنات حية دقيقة.,أتعرف البكتيريا بأنها كائنات حية دقيقة؟
4,التعليم,الهواء يتكون أساساً من النيتروجين.,أيتكون الهواء أساساً من النيتروجين؟


### Duplicate Handling
---



#### Step 1 — Check for harakat-based duplicates (same after stripping tashkeel):

In [9]:
# Strip tashkeel from Question column temporarily to detect harakat-masked duplicates
df['_q_stripped'] = df['Question'].apply(araby.strip_tashkeel)

harakat_dupes = df[df.duplicated(subset=['_q_stripped'], keep=False) &
                   ~df.duplicated(subset=['Question'],   keep=False)]

print(f"Rows that are duplicates ONLY after stripping harakat (not exact): {len(harakat_dupes)}")
if len(harakat_dupes) > 0:
    print("Sample:")
    print(harakat_dupes[['Question', '_q_stripped']].head(10))
else:
    print("All detected duplicates are true exact duplicates — safe to remove.")


Rows that are duplicates ONLY after stripping harakat (not exact): 50
Sample:
                                               Question  \
186   اعربْ الجملةَ التاليةَ: "يذهبُ الطالبُ إلى الم...   
191   اعربْ الجملةَ التاليةَ: "الأطفالُ يلعبونَ في ا...   
207   اعربْ الجملة التالية: "يذهبُ الطالبُ إلى المدر...   
220   اعرب الجملة التالية: "الأطفال يلعبون في الحديقة."   
361                      فيم تكمن أهمية التعليم العالي؟   
451                 فيم تكمن أهمية الابتكار في الصناعة؟   
557                        إلامَ تُركزُ في وقتِ الفراغ؟   
573                       إلامَ تُركزُ في وقتِ الفراغِ؟   
1023                        بمَ يمكن قياس درجة الحرارة؟   
1211                             ماذا تفضلُ من الأطعمة؟   

                                            _q_stripped  
186     اعرب الجملة التالية: "يذهب الطالب إلى المدرسة."  
191   اعرب الجملة التالية: "الأطفال يلعبون في الحديقة."  
207     اعرب الجملة التالية: "يذهب الطالب إلى المدرسة."  
220   اعرب الجملة التالية: "الأطفال يلعب

#### Step 2 — Remove exact raw duplicates:

In [10]:
before = len(df)
df.drop_duplicates(subset=['Question'], inplace=True)
df.reset_index(drop=True, inplace=True)
df.drop(columns=['_q_stripped'], inplace=True)

print(f"Before: {before} rows")
print(f"After : {len(df)} rows")
print(f"Removed: {before - len(df)} exact duplicate rows")


Before: 5009 rows
After : 4645 rows
Removed: 364 exact duplicate rows


### Preprocessing Pipeline
---

#### 1- Remove Noise (keep Arabic characters only):

In [11]:
def remove_noise(text):
    # Keep Arabic letters, Arabic-Indic digits, and whitespace only
    return re.sub(r'[^؀-ۿ\s]', '', str(text))

df['cleaned_question'] = df['Question'].apply(remove_noise)
df[['Question', 'cleaned_question']].head(3)


,Question,cleaned_question
0,أيهما أفضل الدراسة في السابق أم في الوقت الحالي؟,أيهما أفضل الدراسة في السابق أم في الوقت الحالي؟
1,أليس القطن عماد الثروة في مصر؟,أليس القطن عماد الثروة في مصر؟
2,أتصعد الشمس من الشرق؟,أتصعد الشمس من الشرق؟


#### 2- Remove Tashkeel (diacritics):

In [12]:
df['cleaned_question'] = df['cleaned_question'].apply(araby.strip_tashkeel)
df['cleaned_question'].head(3)


,cleaned_question
0,أيهما أفضل الدراسة في السابق أم في الوقت الحالي؟
1,أليس القطن عماد الثروة في مصر؟
2,أتصعد الشمس من الشرق؟


#### 3- Remove Tatweel (elongation character ـ):

In [13]:
df['cleaned_question'] = df['cleaned_question'].apply(araby.strip_tatweel)
df['cleaned_question'].head(3)


,cleaned_question
0,أيهما أفضل الدراسة في السابق أم في الوقت الحالي؟
1,أليس القطن عماد الثروة في مصر؟
2,أتصعد الشمس من الشرق؟


#### 4- Normalize Alef (أ إ آ → ا):

In [14]:
def normalize_alef(text):
    return re.sub(r'[آأإٱ]', 'ا', text)

df['cleaned_question'] = df['cleaned_question'].apply(normalize_alef)
df['cleaned_question'].head(3)


,cleaned_question
0,ايهما افضل الدراسة في السابق ام في الوقت الحالي؟
1,اليس القطن عماد الثروة في مصر؟
2,اتصعد الشمس من الشرق؟


#### 5- Remove Stopwords:

In [15]:
arabic_stopwords = set(stopwords.words('arabic'))

def remove_stopwords(text):
    words = text.split()
    return ' '.join([w for w in words if w not in arabic_stopwords])

df['cleaned_question'] = df['cleaned_question'].apply(remove_stopwords)
df['cleaned_question'].head(3)


,cleaned_question
0,ايهما افضل الدراسة السابق ام الوقت الحالي؟
1,اليس القطن عماد الثروة مصر؟
2,اتصعد الشمس الشرق؟


#### 6- Tokenization:

In [16]:
df['tokens'] = df['cleaned_question'].apply(word_tokenize)
df['tokens'].head(3)


,tokens
0,"[ايهما, افضل, الدراسة, السابق, ام, الوقت, الحا..."
1,"[اليس, القطن, عماد, الثروة, مصر؟]"
2,"[اتصعد, الشمس, الشرق؟]"


#### 7- Remove Punctuation / Non-Arabic tokens:



In [17]:
def clean_tokens(tokens):
    result = []
    for t in tokens:
        # Strip anything that is not an Arabic letter or Arabic-Indic digit
        t_clean = re.sub(r'[^ء-غف-ي٠-٩]', '', t)
        if t_clean:          # keep only non-empty tokens
            result.append(t_clean)
    return result

df['tokens'] = df['tokens'].apply(clean_tokens)
df['tokens'].head(3)


,tokens
0,"[ايهما, افضل, الدراسة, السابق, ام, الوقت, الحالي]"
1,"[اليس, القطن, عماد, الثروة, مصر]"
2,"[اتصعد, الشمس, الشرق]"


#### 8- Stemming (ISRIStemmer):



In [18]:
stemmer = ISRIStemmer()

def stem_tokens(tokens):
    return [stemmer.stem(t) if len(t) >= 4 else t for t in tokens]

df['stemmed_tokens'] = df['tokens'].apply(stem_tokens)
df['stemmed_tokens'].head(3)


,stemmed_tokens
0,"[ايه, فضل, درس, سبق, ام, وقت, الحالي]"
1,"[الس, قطن, عمد, ثرة, مصر]"
2,"[صعد, شمس, شرق]"


#### 9- Join tokens back to string:

In [19]:
df['stemmed_question_text'] = df['stemmed_tokens'].apply(lambda t: ' '.join(t))
df[['Question', 'stemmed_question_text']].head(5)


,Question,stemmed_question_text
0,أيهما أفضل الدراسة في السابق أم في الوقت الحالي؟,ايه فضل درس سبق ام وقت الحالي
1,أليس القطن عماد الثروة في مصر؟,الس قطن عمد ثرة مصر
2,أتصعد الشمس من الشرق؟,صعد شمس شرق
3,أتعرف البكتيريا بأنها كائنات حية دقيقة؟,عرف كتر بان كئن حية دقق
4,أيتكون الهواء أساساً من النيتروجين؟,ايت هوء سسا ترج


#### 10- Second Deduplication on Stemmed Text:



In [20]:
before = len(df)
df.drop_duplicates(subset=['stemmed_question_text'], inplace=True)
df.reset_index(drop=True, inplace=True)

print(f"Before: {before} rows")
print(f"After : {len(df)} rows")
print(f"Normalisation duplicates removed: {before - len(df)}")


Before: 4645 rows
After : 4540 rows
Normalisation duplicates removed: 105


### Dataset Summary
---

In [21]:
print(f"Final dataset shape : {df.shape}")
print(f"Unique categories   : {df['Category'].nunique()}")
print(f"\nCategory distribution:")
print(df['Category'].value_counts())


Final dataset shape : (4540, 7)
Unique categories   : 17

Category distribution:
Category
الثقافة             708
التعليم             572
العلوم              533
الصحة               531
البيئة والطاقة      459
التكنولوجيا         420
التاريخ             358
الاقتصاد والعمل     267
السياسة والقانون    186
الجغرافيا           178
الدين               114
الرياضة              66
البيولوجيا           46
الترفيه              37
السفر والسياحة       33
التطوع               24
علم الاجتماع          8
Name: count, dtype: int64


In [22]:
# Preview final cleaned data
output_cols = ['Question', 'Answer', 'Category', 'cleaned_question', 'stemmed_question_text']
df[output_cols].head(10)


,Question,Answer,Category,cleaned_question,stemmed_question_text
0,أيهما أفضل الدراسة في السابق أم في الوقت الحالي؟,الدراسة في الوقت الحالي تعتبر أفضل بسبب توفر ا...,التعليم,ايهما افضل الدراسة السابق ام الوقت الحالي؟,ايه فضل درس سبق ام وقت الحالي
1,أليس القطن عماد الثروة في مصر؟,القطن يعتبر من أهم المنتجات الزراعية في مصر، و...,الاقتصاد والعمل,اليس القطن عماد الثروة مصر؟,الس قطن عمد ثرة مصر
2,أتصعد الشمس من الشرق؟,الشمس تصعد من الشرق.,التعليم,اتصعد الشمس الشرق؟,صعد شمس شرق
3,أتعرف البكتيريا بأنها كائنات حية دقيقة؟,البكتيريا تُعرف بأنها كائنات حية دقيقة.,التعليم,اتعرف البكتيريا بانها كائنات حية دقيقة؟,عرف كتر بان كئن حية دقق
4,أيتكون الهواء أساساً من النيتروجين؟,الهواء يتكون أساساً من النيتروجين.,التعليم,ايتكون الهواء اساسا النيتروجين؟,ايت هوء سسا ترج
5,أيفرز البنكرياس الأنسولين؟,يفرز البنكرياس الأنسولين.,الصحة,ايفرز البنكرياس الانسولين؟,فرز كرياس سول
6,أيتكون الماء من الهيدروجين والأوكسجين؟,الماء يتكون من الهيدروجين والأوكسجين.,التعليم,ايتكون الماء الهيدروجين والاوكسجين؟,ايت ماء هيدروج كسج
7,أيوجد فيتامين سي في البرتقال؟,فيتامين سي يوجد في البرتقال.,الصحة,ايوجد فيتامين سي البرتقال؟,وجد يتم سي رتقال
8,أيتنفس السمك باستخدام الخياشيم؟,يتنفس السمك باستخدام الخياشيم.,التعليم,ايتنفس السمك باستخدام الخياشيم؟,نفس سمك باستخدام خياشيم
9,أيمكن للنباتات أن تقوم بعملية البناء الضوئي؟,يمكن للنباتات أن تقوم بعملية البناء الضوئي.,التعليم,ايمكن للنباتات ان تقوم بعملية البناء الضوئي؟,ايم نبت ان تقم عمل بنء ضوئ


In [23]:
# Drop categories with fewer than 20 samples (too few to learn)
counts    = df['Category'].value_counts()
drop_cats = counts[counts < 20].index.tolist()

print(f"Dropping {len(drop_cats)} categories: {drop_cats}")
df = df[~df['Category'].isin(drop_cats)].reset_index(drop=True)
print(f"Remaining rows      : {len(df)}")
print(f"Remaining categories: {df['Category'].nunique()}")
print(df['Category'].value_counts())

Dropping 1 categories: ['علم الاجتماع']
Remaining rows      : 4532
Remaining categories: 16
Category
الثقافة             708
التعليم             572
العلوم              533
الصحة               531
البيئة والطاقة      459
التكنولوجيا         420
التاريخ             358
الاقتصاد والعمل     267
السياسة والقانون    186
الجغرافيا           178
الدين               114
الرياضة              66
البيولوجيا           46
الترفيه              37
السفر والسياحة       33
التطوع               24
Name: count, dtype: int64


In [24]:
df = df[df['stemmed_question_text'].str.split().apply(len) >= 2].reset_index(drop=True)

#### 11- Save Preprocessed Dataset:

In [26]:
df[output_cols].to_csv('AAFAQ_Preprocessed.csv', index=False, encoding='utf-8-sig')
print("Saved: AAFAQ_Preprocessed.csv")
print(f"Shape: {df[output_cols].shape}")


Saved: AAFAQ_Preprocessed.csv
Shape: (4522, 5)
